In [0]:
dbutils.widgets.text("race_results_max_ingestion_date","")
race_results_max_ingestion_date=dbutils.widgets.get("race_results_max_ingestion_date")

In [0]:
#dbutils.widgets.text("drivers_qualifying_max_ingestion_date","")
drivers_qualifying_max_ingestion_date=dbutils.widgets.get("drivers_qualifying_max_ingestion_date")

In [0]:
class Gold_race_wise_analysis():
    main_path="/Volumes/formula1_race/default/formula1/"
    gold_path = "formula1_race_project/gold"
    silver_path = "formula1_race_project/silver"

    def __init__(self,table,race_result_max_ingestion_date,drivers_qualifying_max_ingestion_date):
         self.table=table
         #latest water mark value for race_results table from Gold_race_results class
         self.race_result_max_ingestion_date=race_result_max_ingestion_date 
         #latest water mark value for race_results table from Gold_drivers_qualifying class
         self.drivers_qualifying_max_ingestion_date=drivers_qualifying_max_ingestion_date  

    def read_input(self):
        from pyspark.sql.functions import max,col,expr,count,to_timestamp,lit,try_to_timestamp
        race_year_list=list()
        #printing last water mark value of race_results table
        print(f"race_result_max_ingestion_date:{self.race_result_max_ingestion_date}")
        print(f"race_result_max_ingestion_date:{type(self.race_result_max_ingestion_date)}")

        #fetching incremental race_results data from gold table race_results
        if spark.catalog.tableExists("formula1_race.silver.race_results"):
            Incr_race_results_df= (spark.read.table('formula1_race.silver.race_results')
                                  .filter(col('results_ingestion_date')>self.race_result_max_ingestion_date))
            
            #fetching distinct race_year from incremental race_results data
            Incr_race_results_df_list = (Incr_race_results_df
                                   .select(col('race_year')).distinct()
                                   .orderBy(col('race_year').asc()).collect()
                                   )
            #fetching incremental race_results data and printing count of records
            print("race_wise_analysis:Incr_race_results_df batch count")
            display(Incr_race_results_df.select(count('*')))
            race_results_race_year_list=[r.race_year for r in Incr_race_results_df_list]

        #listing of distinct race_years of incremental race_results and printing the list
        print(f"race_year_list:{race_results_race_year_list}")


        #####################################################################
        print(f"drivers_qualifying_max_ingestion_date:{self.drivers_qualifying_max_ingestion_date}")
        print(f"drivers_qualifying_max_ingestion_date:{type(self.drivers_qualifying_max_ingestion_date)}")
        #ffetching incremental drivers_qualifying data from gold table drivers_qualifying
        if spark.catalog.tableExists("formula1_race.silver.drivers_qualifying"):
            Incr_drivers_qualifying_df= (spark.read.table('formula1_race.silver.drivers_qualifying')
                                  .filter(col('qualifying_ingestion_date')>self.drivers_qualifying_max_ingestion_date))
            
            #fetching distinct race_year from incremental drivers_qualifying data
            Incr_drivers_qualifying_df_list = (Incr_drivers_qualifying_df
                                   .select(col('race_year')).distinct()
                                   .orderBy(col('race_year').asc()).collect()
                                   )
            #fetching incremental drivers_qualifying data and printing count of records
            print("race_wise_analysis:Incr_drivers_qualifying_df batch count")
            display(Incr_drivers_qualifying_df.select(count('*')))
            drivers_qualifying_race_year_list=[r.race_year for r in Incr_drivers_qualifying_df_list]

        #listing of distinct race_years of incremental drivers_qualifying and printing the list
        print(f"race_year_list:{drivers_qualifying_race_year_list}")
        race_year_list=[race_results_race_year_list,drivers_qualifying_race_year_list]
        return race_year_list
    
    def apply_transformations(self,race_year_list):
        from pyspark.sql.functions import round,col,broadcast,dense_rank,avg,expr,sum,when,concat,lit,count,max
        from pyspark.sql.window import Window
        cc_spec_window=Window.partitionBy("driver_name").orderBy(col("race_year").asc())
        rn_spec_window=Window.partitionBy("race_year","driver_name").orderBy(col("race_name").asc())
        
        #fetching only required data from race_results table which is required for aggregation and printing the count of records
        Incr_race_results_df= (spark.read.table('formula1_race.silver.race_results')
                               .filter(col('race_year').isin(race_year_list[0]))
                               )
        print("race_wise_analysis:Incr_race_results_df original count")
        display(Incr_race_results_df.select(count('*')))
        
        #find aggregated total_points for each driver in each race_year(race_results table)
        cte_df=(Incr_race_results_df.groupBy(col("race_year"),col("driver_name"))
                .agg(sum(col("result_points")).alias("total_points"))
                .select(col("race_year").alias('cte_race_year'),
                        col("driver_name").alias('cte_driver_name'),
                        col("total_points"))
                )
        # find rolling points for each driver in each race_year(race_results table)
        cte1_df=(Incr_race_results_df
                 .withColumn("rolling_points_race_year",sum(col("result_points")).over(cc_spec_window))
                 .withColumn("rolling_points_race_name",sum(col("result_points")).over(rn_spec_window))
                 .withColumn("max_fastest_lap_time",max(col('result_fastest_lap_time'))
                                                .over(Window.partitionBy(col('race_year'),col('race_name'),col('driver_name'))))
                 .select("*")
                )
        #fetching only required data from drivers_qualifying table which is required for joining and printing the count of records
        if(len(race_year_list[1]) !=0): 
            cte2_df=(spark.read.table('formula1_race.silver.drivers_qualifying')
                                .filter(col('race_year').isin(race_year_list[1]))
                                .withColumnRenamed("race_year","qualifying_race_year")
                                .withColumnRenamed("race_name","qualifying_race_name")
                                .withColumnRenamed("driver_name","qualifying_driver_name")  
                                )
        else:#if there is no incremental data in drivers_qualifying table then fetch drivers_qualifying recordes using race_years of race_results table(according to join condition it will be possible)
            cte2_df=(spark.read.table('formula1_race.silver.drivers_qualifying')
                                .filter(col('race_year').isin(race_year_list[0]))
                                .withColumnRenamed("race_year","qualifying_race_year")
                                .withColumnRenamed("race_name","qualifying_race_name")
                                .withColumnRenamed("driver_name","qualifying_driver_name")  
                                )

        print("race_wise_analysis:Incr_drivers_qualifying_df original count")
        display(cte2_df.select(count('*')))

        #aliasing the tables before join
        cte_df = cte_df.alias("cte") # aggreation applied on incremental race_results data
        cte1_df = cte1_df.alias("cte1") # transformation applied on incremental race_results data
        cte2_df=cte2_df.alias("cte2")# fetch requried columns from incremental drivers_qualifying data
        
        #performing join operation with all the tables
        join_df=(cte_df.join(cte1_df,[cte_df.cte_race_year==cte1_df.race_year,
                                      cte_df.cte_driver_name==cte1_df.driver_name],how="inner")
                        .join(cte2_df,[cte1_df.race_year==cte2_df.qualifying_race_year,
                                       cte1_df.race_name==cte2_df.qualifying_race_name,
                                      cte1_df.driver_name==cte2_df.qualifying_driver_name],how="inner")
                  .withColumn("position",dense_rank().over(Window.partitionBy(col("race_year")).orderBy(col("total_points").desc())))
                  .select([col("cte1." + c) for c in cte1_df.columns] + [col("cte.total_points"),col("position")]+[col("qualifying_q1"),col("qualifying_q2"),col("qualifying_q3")])
                  )#for fetching all columns from cte1 used different approach in select
        display(join_df.filter(col("race_year")==2018))
        return join_df
    
    def write_output(self,apply_tran_df):
        # writing those data into gold layer table by partitioning according to filter approache using dynamic partitionOverwriteMode as true with overwritte mode
        (apply_tran_df.write.option("partitionOverwriteMode", "dynamic")
         .partitionBy("race_year").mode("overwrite")
         .saveAsTable(f"formula1_race.gold.{self.table}"))
        print("final_count_in table")
        display(spark.sql(f'select count(*) from formula1_race.gold.{self.table}'))
        print("Data write into gold race_wise_analysis table is Done")

    def process(self):
        print("Started gold-ingestion-race_wise_analysis in runing....")
        race_year_list=self.read_input()
        apply_tran_df=self.apply_transformations(race_year_list)
        self.write_output(apply_tran_df)
                  
        


In [0]:
Gold_race_wise_analysis_instance = Gold_race_wise_analysis("race_wise_analysis",race_results_max_ingestion_date,drivers_qualifying_max_ingestion_date)
Gold_race_wise_analysis_instance .process()
print("Successfully Gold_race_wise_analysis is ran")